In [0]:
from pyspark.sql.functions import col, when, sum, count, round

In [0]:
absence_df = spark.read.table("nba_insights.silver.games_stats_with_stars_absence")
franchise_history_df = spark.read.table("nba_insights.silver.star_franchise_games_history") 
stars_df = spark.read.table("nba_insights.silver.star_players")

In [0]:
print(f"Absence table count: {absence_df.count()}")
print(f"Franchise history count: {franchise_history_df.count()}")
print(f"Stars count: {stars_df.count()}")

In [0]:
# Determine if the team won or lost when the star was absent
outcomes_df = absence_df.withColumn(
    "team_outcome",
    when(col("star_team_id") == col("team_id_home"), col("wl_home"))
    .when(col("star_team_id") == col("team_id_away"), col("wl_away"))
    .otherwise(None)
)

# Group by player to get total absence wins and games missed
absence_agg_df = outcomes_df.groupBy("player_id").agg(
    count("game_id").alias("games_absent"),
    sum(when(col("team_outcome") == "W", 1).otherwise(0)).alias("absence_wins")
)

display(absence_agg_df)

In [0]:
# Group the franchise history table to get the team's total performance during the star's era
totals_agg_df = franchise_history_df.groupBy("player_id").agg(
    count("game_id").alias("total_team_games"),
    sum(when(col("team_outcome") == "W", 1).otherwise(0)).alias("total_team_wins")
)

display(totals_agg_df)

In [0]:
# Join the totals with the absences
delta_df = totals_agg_df.join(absence_agg_df, on="player_id", how="left").fillna(0)

# Deduct absences to find "Presence" stats
final_math_df = delta_df.withColumn(
    "games_present", col("total_team_games") - col("games_absent")
).withColumn(
    "presence_wins", col("total_team_wins") - col("absence_wins")
)

# Calculate Percentages and the Impact Delta
pct_df = final_math_df.withColumn(
    "win_pct_with_star", round(col("presence_wins") / col("games_present"), 3)
).withColumn(
    "win_pct_without_star", 
    when(col("games_absent") > 0, round(col("absence_wins") / col("games_absent"), 3)).otherwise(0.0)
).withColumn(
    "impact_delta", round(col("win_pct_with_star") - col("win_pct_without_star"), 3)
)

In [0]:
# Join player names for business readability
gold_star_impact = pct_df.join(
    stars_df.select("player_id", "display_first_last"), 
    on="player_id"
).select(
    "player_id", 
    "display_first_last",
    "games_present", 
    "win_pct_with_star",
    "games_absent", 
    "win_pct_without_star",
    "impact_delta"
) # Remove statistical noise (e.g., players with < 15 absences tracked)

# Display the data so you can build Databricks visualizations directly on it
display(gold_star_impact)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Convert PySpark DataFrame to Pandas for local plotting
# We filter for top players to keep the graphs readable
gold_pd = gold_star_impact.toPandas()

# Sort by impact_delta for our charts
gold_pd = gold_pd.sort_values('impact_delta', ascending=False)
top_stars = gold_pd.head(15) # Top 15 most impactful stars

In [0]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_stars, 
    x='impact_delta', 
    y='display_first_last', 
    palette='viridis'
)

plt.title("The Superstar Impact: Win Percentage Delta (Top 15)", fontsize=14, fontweight='bold')
plt.xlabel("Win % Delta (With Star - Without Star)", fontsize=12)
plt.ylabel("")
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Show the plot in the Databricks cell
plt.show()

In [0]:
# Set up the grouped bar chart data
x = np.arange(len(top_stars['display_first_last']))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, top_stars['win_pct_with_star'], width, label='With Star', color='#1f77b4')
rects2 = ax.bar(x + width/2, top_stars['win_pct_without_star'], width, label='Without Star', color='#ff7f0e')

ax.set_ylabel('Win Percentage', fontsize=12)
ax.set_title('Team Win Percentage: With vs. Without Superstar', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(top_stars['display_first_last'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(10, 6))

# Plot all players, not just the top 15
sns.scatterplot(
    data=gold_pd, 
    x='games_absent', 
    y='impact_delta', 
    size='games_present', # Size of the bubble is how many games they actually played
    sizes=(50, 400),
    alpha=0.6,
    color='#2ca02c'
)

# Add a horizontal line at 0 (meaning no impact)
plt.axhline(0, color='red', linestyle='--')

plt.title("Statistical Confidence: Impact vs. Games Missed", fontsize=14, fontweight='bold')
plt.xlabel("Total Games Absent (Sample Size)", fontsize=12)
plt.ylabel("Win % Impact Delta", fontsize=12)

# Annotate the extreme outliers so the audience knows who they are
for idx, row in gold_pd.nlargest(3, 'impact_delta').iterrows():
    plt.text(row['games_absent'] + 2, row['impact_delta'], row['display_first_last'], fontsize=9)

plt.grid(True, linestyle='--', alpha=0.5)
plt.show()